In [ ]:
# ==== 0) 환경 준비 ====
# 원인: 모델 실행 및 데이터 처리에 필요한 라이브러리와 환경을 설정합니다.
# 과정: os, random, math, pandas, torch, transformers 등 필수 라이브러리를 가져옵니다.
# 결과: 코드 실행에 필요한 모든 도구와 데이터가 메모리에 로드되어 준비됩니다.
import os, random, math
import pandas as pd
import torch
import transformers

# --- 모델 및 토크나이저 로드 ---
# 원인: 텍스트를 의미적 벡터로 변환하기 위해 사전 학습된 언어 모델이 필요합니다.
# 과정: Hugging Face Hub에서 'jxm/cde-small-v2' 모델과 'bert-base-uncased' 토크나이저를 로드합니다.
#      'jxm/cde-small-v2'는 CDE(Context-Denoising Encoder) 모델로, 데이터셋의 일부(minicorpus)를 참조하여
#      문맥에 맞는 특화된 임베딩을 생성하는 2단계 임베딩 방식을 사용합니다.
# 결과: model과 tokenizer 객체가 생성되어 텍스트를 토큰화하고 임베딩할 준비를 마칩니다.
model_name = "jxm/cde-small-v2"
model = transformers.AutoModel.from_pretrained(model_name, trust_remote_code=True)
tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

# --- 장치 설정 및 모델 평가 모드 전환 ---
# 원인: 계산 속도 향상을 위해 GPU를 우선 사용하고, 추론 시에는 불필요한 연산을 비활성화해야 합니다.
# 과정: torch.cuda.is_available()로 GPU 사용 가능 여부를 확인하고 device를 설정합니다.
#      model.to(device)로 모델을 해당 장치로 옮기고, model.eval()과 torch.set_grad_enabled(False)로
#      추론 모드로 전환하여 메모리 사용량을 줄이고 속도를 높입니다.
# 결과: 모델이 최적화된 추론 환경으로 설정됩니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
torch.set_grad_enabled(False)

# --- 프리픽스(접두사) 정의 ---
# 원인: CDE 모델은 입력이 '질의(query)'인지 '문서(document)'인지 구분하기 위해 특별한 텍스트 접두사를 사용하도록 학습되었습니다.
# 과정: 'search_query: '와 'search_document: ' 문자열을 변수에 할당합니다.
# 결과: 이후 텍스트를 토큰화할 때 이 접두사들을 앞에 붙여 모델에 입력의 종류를 알려줍니다.
query_prefix      = "search_query: "
document_prefix = "search_document: "

# --- 데이터 준비 및 미니코퍼스(Minicorpus) 생성 ---
# 원인: CDE 모델의 1단계 임베딩을 위해 전체 데이터셋의 특성을 대표하는 작은 샘플 데이터셋(minicorpus)이 필요합니다.
# 과정: 1. 모델 설정에서 권장하는 minicorpus 크기를 가져옵니다.
#      2. Pandas를 사용해 학습 및 검증 데이터를 CSV 파일에서 읽어옵니다.
#      3. 학습 문서(train_docs)에서 minicorpus 크기만큼 무작위로 샘플링합니다. 문서가 부족하면 반복해서 채웁니다.
#      4. 생성된 minicorpus의 각 문서에 'document_prefix'를 붙여 토큰화합니다.
# 결과: 전체 데이터셋의 문맥 정보를 담은 'mc_tok'(토큰화된 미니코퍼스) 텐서가 생성됩니다.
minicorpus_size = getattr(model.config, "transductive_corpus_size", 512)
print("CDE v2 minicorpus size:", minicorpus_size)

df_train = pd.read_csv("/home/alpaco/sryang/Training.csv")
df_valid = pd.read_csv("/home/alpaco/sryang/validation.csv")

train_docs = df_train.iloc[:,0].dropna().astype(str).tolist()
val_queries = df_valid.iloc[:,0].dropna().astype(str).tolist()

# 학습 문서가 minicorpus 크기보다 크면 무작위 샘플링, 작으면 반복해서 채움
if len(train_docs) >= minicorpus_size:
    minicorpus_docs = random.sample(train_docs, k=minicorpus_size)
else:
    reps = math.ceil(minicorpus_size / max(1, len(train_docs)))
    minicorpus_docs = (train_docs * reps)[:minicorpus_size]

# 미니코퍼스를 토큰화
mc_tok = tokenizer(
    [document_prefix + d for d in minicorpus_docs],
    truncation=True, padding=True, max_length=768, return_tensors="pt"
).to(device)

# ==== 1단계 임베딩: 데이터셋 문맥 벡터 생성 ====
# 원인: 문서와 질의를 임베딩하기 전에, CDE 모델이 참조할 '데이터셋 전체의 문맥'을 나타내는 벡터가 필요합니다.
# 과정: 토큰화된 미니코퍼스(mc_tok)를 배치(batch) 단위로 나누어 모델의 'first_stage_model'에 입력합니다.
#      배치 처리는 GPU 메모리 부족 문제를 방지합니다.
# 결과: 데이터셋 전체의 의미적 특성을 요약한 'dataset_embeddings' 텐서가 생성됩니다. 이 벡터는 2단계의 '가이드' 역할을 합니다.
batch_size = 32
dataset_embeddings = []
for i in range(0, mc_tok["input_ids"].size(0), batch_size):
    batch = {k: v[i:i+batch_size] for k, v in mc_tok.items()}
    with torch.no_grad():
        emb = model.first_stage_model(**batch)
    dataset_embeddings.append(emb)
dataset_embeddings = torch.cat(dataset_embeddings, dim=0).to(device)

# ==== 2단계 (문서): 문맥 정보를 활용한 최종 문서 임베딩 ====
# 원인: 각 문서를 '데이터셋 전체 문맥' 안에서 더 정교한 의미 벡터로 표현해야 합니다.
# 과정: 1. 전체 학습 문서(train_docs)를 토큰화합니다.
#      2. 배치 단위로 나누어 모델의 'second_stage_model'에 입력합니다.
#      3. 이때, 각 문서의 토큰 정보와 함께 **1단계에서 만든 'dataset_embeddings'를 함께** 입력합니다.
#      4. 생성된 임베딩 벡터를 normalize(정규화)하여 길이를 1로 만듭니다. 이는 유사도 계산의 안정성을 높입니다.
# 결과: 데이터셋의 특성에 맞게 '보정된' 최종 문서 임베딩('doc_embeddings') 텐서가 생성됩니다.
docs_tok = tokenizer(
    [document_prefix + d for d in train_docs],
    truncation=True, padding=True, max_length=768, return_tensors="pt"
).to(device)

doc_embeddings = []
for i in range(0, docs_tok["input_ids"].size(0), batch_size):
    batch = {k: v[i:i+batch_size] for k, v in docs_tok.items()}
    with torch.no_grad():
        emb = model.second_stage_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            dataset_embeddings=dataset_embeddings # 1단계에서 생성한 문맥 벡터를 함께 전달
        )
        emb = torch.nn.functional.normalize(emb, p=2, dim=1) # L2 정규화
    doc_embeddings.append(emb)
doc_embeddings = torch.cat(doc_embeddings, dim=0).to(device)

# ==== 2단계 (질의): 문맥 정보를 활용한 최종 질의 임베딩 ====
# 원인: 질의(query) 또한 문서와 동일한 의미 공간(semantic space) 상에서 벡터로 표현되어야 정확한 비교가 가능합니다.
# 과정: 문서 임베딩과 완전히 동일한 과정을 거칩니다.
#      1. 검증용 질의(val_queries)를 토큰화합니다.
#      2. 배치 단위로 나누어 모델의 'second_stage_model'에 입력합니다.
#      3. 이때도 **동일한 'dataset_embeddings'를 함께** 입력하여 문서와 같은 문맥을 공유하도록 합니다.
#      4. 생성된 임베딩을 정규화합니다.
# 결과: 문서 임베딩과 같은 공간 상에 표현되는 최종 질의 임베딩('query_embeddings') 텐서가 생성됩니다.
q_tok = tokenizer(
    [query_prefix + q for q in val_queries],
    truncation=True, padding=True, max_length=512, return_tensors="pt"
).to(device)

query_embeddings = []
for i in range(0, q_tok["input_ids"].size(0), batch_size):
    batch = {k: v[i:i+batch_size] for k, v in q_tok.items()}
    with torch.no_grad():
        emb = model.second_stage_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            dataset_embeddings=dataset_embeddings # 동일한 문맥 벡터를 전달
        )
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    query_embeddings.append(emb)
query_embeddings = torch.cat(query_embeddings, dim=0).to(device)

print("문서 임베딩 크기:", doc_embeddings.shape)
print("질의 임베딩 크기:", query_embeddings.shape)

A new version of the following files was downloaded from https://huggingface.co/jxm/cde-small-v2:
- model.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Disabled 23 dropout modules from model type <class 'transformers_modules.jxm.cde-small-v2.4e1d021a6c3fd7ce8aa0a7204057eee5ae61d390.model.BiEncoder'>
Disabled 46 dropout modules from model type <class 'transformers_modules.jxm.cde-small-v2.4e1d021a6c3fd7ce8aa0a7204057eee5ae61d390.model.ContextualDocumentEmbeddingTransformer'>
CDE v2 minicorpus size: 512


/home/alpaco/anaconda3/envs/soray/lib/python3.10/site-packages/torch/_inductor/compile_fx.py:282: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


문서 임베딩 크기: torch.Size([51628, 768])
질의 임베딩 크기: torch.Size([6640, 768])


In [ ]:
torch.save(dataset_embeddings.cpu(), "/home/alpaco/sryang/embedding_result/cde_minicorpus.pt")
torch.save(doc_embeddings.cpu(), "/home/alpaco/sryang/embedding_result/cde_doc.pt")
torch.save(query_embeddings.cpu(), "/home/alpaco/sryang/embedding_result/cde_query_emb.pt")